In [1]:
import pandas as pd
import numpy as np
import wandb
import os
import yaml
import torch

from transformers import AutoTokenizer, BartForConditionalGeneration
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, EarlyStoppingCallback
from torch.utils.data import Dataset
from rouge import Rouge
from tqdm import tqdm


/root/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Config File Load

In [2]:
# Config 파일 불러오기
config_path = './config.yaml'
with open(config_path, "r", encoding='utf-8') as file:
    config = yaml.safe_load(file)

print("Config Load 완료 세팅된 주요 값 확인")
print(f"- Enocder Max Lenth (대화문 95% 커버) : {config['tokenizer']['encoder_max_len']}")
print(f"- Decoder Max Lenth (요약문 커버) : {config['tokenizer']['decoder_max_len']}")
print(f"- 발굴된 특수 토큰 개수 : {len(config['tokenizer']['special_tokens'])}개")

Config Load 완료 세팅된 주요 값 확인
- Enocder Max Lenth (대화문 95% 커버) : 1024
- Decoder Max Lenth (요약문 커버) : 50
- 발굴된 특수 토큰 개수 : 22개


## Model Load & 임베딩 Layer 확장

📒 전략 <br>
- 우리가 허깅페이스에서 이미 학습된 Ko-BART 모델을 불러온건데, 거기에는 우리 데이터에서 정제한 데이터 내용이 없음
- 그래서 모델의 임베딩 레이어 크기를 새 단어장 크기만큼 물리적으로 늘려야함 -> 결국엔 사전에 우리 데이터를 추가하는 로직임

In [3]:
model_name = config['general']['model_name']
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"💻 Loding Device : {device}")

💻 Loding Device : cuda:0


In [4]:
# 1. 모델이랑 토크나이저 불러오기
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

print(f"특수 토큰 추가 전 Tokenizer 단어장 크기 : {len(tokenizer)}")

# 2. 우리가 찾았던 22개 특수 토큰을 Tokenizer 사전에 등록
special_tokens_dict = {'additional_special_tokens': config['tokenizer']['special_tokens']}
tokenizer.add_special_tokens(special_tokens_dict)

# 3. 늘어난 단어장 크기에 맞춰서 모델의 임베딩 레이어 (뇌 용량) 확장
model.resize_token_embeddings(len(tokenizer))

# 모델을 GPU 로 이동
model.to(device)

print(f"특수 토큰 추가 후 Tokenizer 단어장 크기 : {len(tokenizer)}")
print("Tokenizer 와 Model Setting 그리고 임베딩 확장 완료")

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 262/262 [00:00<00:00, 20519.62it/s]
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion

특수 토큰 추가 전 Tokenizer 단어장 크기 : 30000
특수 토큰 추가 후 Tokenizer 단어장 크기 : 30022
Tokenizer 와 Model Setting 그리고 임베딩 확장 완료


In [5]:
# 모델이 새롭게 들어와야 새로운 적용값들이 적용되기에 해당 사항 Debug Code
test_text = "[화자1]은 [화자2]에게 감기에 안 걸렸다고 말했다."
tokens = tokenizer.tokenize(test_text)
print(f"토크나이징 테스트 결과 : {tokens}")

토크나이징 테스트 결과 : ['[화자1]', '▁은', '▁', '[화자2]', '▁', '에게', '▁감', '기에', '▁안', '▁걸', '렸다고', '▁말했다.']


## Text -> Number

In [6]:
# 1. 전처리 Class (BART 입력 형태에 맞게 텍스트에 <s>, </s> 토큰을 붙여주는 역할)
class Preprocess:
    def __init__(self, bos_token: str, eos_token: str):
        self.bos_token = bos_token
        self.eos_token = eos_token

    def make_input(self, df):
        # 인코더 입력 : 대화문 원본 (이미 [주제: ...]가 붙어있음)
        encoder_input = df['dialogue'].tolist()

        # 디코더 입력 : <s> + 요약문 (학습 시 모델에게 "여기서부터 요약 시작이다" 라고 알려줌)
        decoder_input = df['summary'].apply(lambda x : self.bos_token + str(x)).tolist()

        # 정답 레이블 : 요약문 + </s> (학습할 때 모델한테 "여기서 문장이 끝남" 이라고 알려줌)
        decoder_output = df['summary'].apply(lambda x : str(x) + self.eos_token).tolist()

        return encoder_input, decoder_input, decoder_output

In [7]:
# 2. PyTorch DataSet 클래스 (모델 <- 데이터 를 하나씩 넣어주기)
class CustomDataset(Dataset):
    def __init__(self, encoder_input, decoder_input, labels, length):
        self.encoder_input = encoder_input
        self.decoder_input = decoder_input
        self.labels = labels
        self.length = length
    
    def __getitem__(self, idx):
        # 텐서 복사, 분리 (안전한 학습을 위해 detach 사용)
        item = {key: torch.tensor(val[idx]) for key, val in self.encoder_input.items()}
        item2 = {key: torch.tensor(val[idx]) for key, val in self.decoder_input.items()}

        # BART 모델의 입력 규격에 맞게 이름 변경
        item['decoder_input_ids'] = item2['input_ids']
        item['decoder_attention_mask'] = item2['attention_mask']
        item['labels'] = torch.tensor(self.labels['input_ids'][idx])

        return item
    
    def __len__(self):
        return self.length

In [ ]:
# 1. Data 불러오기
data_path = config['general']['data_path']

# 원본과 증강 데이터를 각각 불러와서 병합
train_ori = pd.read_csv(os.path.join(data_path, 'train_processed.csv'))
# 증강 데이터 파일명이 다르면 아래 'train_augmented.csv'를 알맞게 수정해!)
train_aug = pd.read_csv(os.path.join(data_path, 'train_augmented.csv')) 

train_df = pd.concat([train_ori, train_aug], ignore_index=True)

# 원본과 증강 데이터가 편향되지 않게 골고루 섞기 (Shuffle)
train_df = train_df.sample(frac=1.0, random_state=42).reset_index(drop=True)

val_df = pd.read_csv(os.path.join(data_path, 'dev_processed.csv'))

print(f"Data Load 완료! (원본+증강) Train: {len(train_df)}개, Val: {len(val_df)}개")

Data Load 완료 Train: 24271개, Val: 499개


In [9]:
# 2. 전처리기 인스턴스 생성 및 입,출력 Text 분리
preprocessor = Preprocess(config['tokenizer']['bos_token'], config['tokenizer']['eos_token'])

# 대화문(인코더 입력), <s> + 요약문(디코더 입력), 요약문 + </s> (정답레이블) 생성
enc_in_train, dec_in_train, dec_out_train = preprocessor.make_input(train_df)
enc_in_val, dec_in_val, dec_out_val = preprocessor.make_input(val_df)

In [ ]:
# 3. 토크나이저를 통해서 텍스트를 숫자(Tensor)로 변환
tok_enc_train = tokenizer(enc_in_train, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['encoder_max_len'])
tok_enc_val = tokenizer(enc_in_val, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['encoder_max_len'])

# 요약문 (Decoder) 토크나이징 (길이 256)
tok_dec_in_train = tokenizer(dec_in_train, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['decoder_max_len'])
tok_dec_out_train = tokenizer(dec_out_train, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['decoder_max_len'])

tok_dec_in_val = tokenizer(dec_in_val, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['decoder_max_len'])
tok_dec_out_val = tokenizer(dec_out_val, return_tensors="pt", padding=True, truncation=True, max_length=config['tokenizer']['decoder_max_len'])


In [11]:
# 4. PyTorch DataSet 객체 생성
train_dataset = CustomDataset(tok_enc_train, tok_dec_in_train, tok_dec_out_train, len(enc_in_train))
val_dataset = CustomDataset(tok_enc_val, tok_dec_in_val, tok_dec_out_val, len(enc_in_val))

## Train Set & WanDB

📒 전략 <br>
- 방어적 평가 함수 : ROUGE => 모델이 학습하는 중간중간에 자기가 얼마나 잘하고 있는지 스스로 채점이 ROUGE <br>
- 이 때, predict_with_generate=True 옵션이 존재하는데 이걸 켜두면 모델이 실제로 텍스트를 생성해서 정답이랑 비교

💡 인사이트로 발생했던 Okt 파괴 테스트 : 형태소 분석기가 [주제:...] 같은 부분을 다 슬라이싱 해버려서 오히려 점수가 깎일 수 있었음
=> 그래서 모델이 생성한 텍스트를 ROUGE 채점에 넘기기 전에 우리가 Config 에 등록한 remove_token([주제: ,] 등등) 를 싹 지워버리는 클렌징폼 코드를 함수로 구현

- FP16 (혼합 정밀도 학습) : Config 에서 fp16: Ture 로 설정해뒀는데, 우리가 시퀀스 길이를 1024 로 2배나 늘렸기 때문에 메모리가 엄청나게 부족할거다.
- 그래서 FP16 은 소수점 아래 숫자의 정밀도를 살짝 낮춰서 (32Bit -> 16Bit) 메모리 사용량을 반으로 줄이고 학습 속도는 2배 이상 끌어 올리는 기술

In [12]:
# 1. ROUGE 평가 함수
def compute_metrics(pred):
    rouge = Rouge()
    predictions = pred.predictions
    labels = pred.label_ids

    # 모델이 예측을 포기한 부분(-100)을 패딩 토큰으로 변경하여 에러 방지
    predictions[predictions == -100] = tokenizer.pad_token_type_id
    labels[labels == -100] = tokenizer.pad_token_id

    # 숫자를 다시 우리가 읽을 수 있는 텍스트로 디코딩
    decoded_preds = tokenizer.batch_decode(predictions, skip_special_tokens=False, clean_up_tokenization_spaces=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=False, clean_up_tokenization_spaces=True)

    # 프롬프트 찌꺼기 및 특수기호 제거 (ROUGE 형태소 분석기 파괴 방지)
    remove_tokens = config['inference']['remove_tokens']
    for token in remove_tokens:
        decoded_preds = [sentence.replace(token, " ") for sentence in decoded_preds]
        decoded_labels = [sentence.replace(token, " ") for sentence in decoded_labels]

    # 최종 ROUGE-1, 2, L 점수 계산 (F1 Socre 기준)
    results = rouge.get_scores(decoded_preds, decoded_labels, avg=True)
    result = {key: value["f"] for key, value in results.items()}

    # 대회 리더보드용 final_result 점수를 로컬에서 흉내 내보기
    result["final_result"] = (result["rouge-1"] + result["rouge-2"] + result["rouge-l"]) / 3.0 * 100.0
    return result

print("ROUGE 평가 및 클렌징 함수 세팅 완료")

# 2. Training Arguments (학습 환경 뼈대 구축)
training_args = Seq2SeqTrainingArguments(
    output_dir=os.path.join(config['general']['output_dir'], "model_save/checkpoints"),
    # overwrite_output_dir=config['training']['overwrite_output_dir'], 
    num_train_epochs=config['training']['num_train_epochs'],
    learning_rate=float(config['training']['learning_rate']),
    per_device_train_batch_size=config['training']['per_device_train_batch_size'], 
    per_device_eval_batch_size=config['training']['per_device_eval_batch_size'],
    warmup_ratio=config['training']['warmup_ratio'],
    weight_decay=config['training']['weight_decay'],
    lr_scheduler_type=config['training']['lr_scheduler_type'],
    optim=config['training']['optim'],
    gradient_accumulation_steps=config['training']['gradient_accumulation_steps'],
    label_smoothing_factor=0.1,     # 추가로직 : 할루시네이션 방지 규제 추가
    eval_strategy=config['training']['evaluation_strategy'],
    save_strategy=config['training']['save_strategy'],
    save_total_limit=config['training']['save_total_limit'],
    fp16=config['training']['fp16'], 
    load_best_model_at_end=config['training']['load_best_model_at_end'],
    seed=config['training']['seed'],
    logging_dir=config['training']['logging_dir'],
    logging_strategy=config['training']['logging_strategy'],
    predict_with_generate=config['training']['predict_with_generate'], 
    generation_max_length=config['training']['generation_max_length'],
    do_train=config['training']['do_train'],
    do_eval=config['training']['do_eval'],
    report_to=config['training']['report_to']
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


ROUGE 평가 및 클렌징 함수 세팅 완료


In [13]:
# 3. W&B 프로젝트 초기화
wandb.init(
    entity=config['wandb']['entity'],
    project=config['wandb']['project'],
    name=config['wandb']['name']
)

# 모델 체크 포인트도 W&B 서버에 백업하도록 설정
os.environ["WANDB_LOG_MODEL"] = "true"
os.environ["WANDB_WATCH"] = "false"

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /data/ephemeral/home/.netrc.
wandb: Currently logged in as: gam10678 (gam10678-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [14]:
# 4. Early Stopping
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=config['training']['early_stopping_patience'],
    early_stopping_threshold=config['training']['early_stopping_threshold']
)

# Train
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[early_stopping]
)

print("완료")

[RANK 0] Detected kernel version 5.4.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
Received unrecognized `WANDB_LOG_MODEL` setting value=true; so disabling `WANDB_LOG_MODEL`


완료


## Train

In [ ]:
# 학습
train_result = trainer.train()

# W&B 세션 종료
wandb.finish()

# SOTA 모델 이랑 토크나이저를 저장
best_model_path = config['general']['output_dir'] + "Model_save/kobart_best_model"
trainer.save_model(best_model_path)
tokenizer.save_pretrained(best_model_path)

print(f"{best_model_path} 에 저장이 완료되었습니다.")

/tmp/ipykernel_658903/2913228691.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item2 = {key: torch.tensor(val[idx]) for key, val in self.decoder_input.items()}
/tmp/ipykernel_658903/2913228691.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels['input_ids'][idx])


Epoch,Training Loss,Validation Loss,Rouge-1,Rouge-2,Rouge-l,Final Result
1,3.544317,2.400729,0.342637,0.112187,0.324633,25.981918
2,2.248504,2.346096,0.347007,0.123992,0.325866,26.562173
3,2.091637,2.331183,0.355867,0.131040,0.337103,27.466989
4,2.004295,2.329477,0.362578,0.136828,0.342019,28.047503
5,1.952722,2.329864,0.360132,0.133751,0.340058,27.798006
6,1.925500,2.334198,0.359114,0.135814,0.339635,27.818759
7,1.915469,2.334441,0.362895,0.135805,0.342063,28.025414


/tmp/ipykernel_658903/2913228691.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encoder_input.items()}
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.28it/s]
/tmp/ipykernel_658903/2913228691.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item2 = {key: torch.tensor(val[idx]) for key, val in self.decoder_input.items()}
/tmp/ipykernel_658903/2913228691.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item['labels'] = torch.tensor(self.labels['input_ids'][id

eval/final_result,▁▃▆█▇▇█
eval/loss,█▃▁▁▁▁▁
eval/rouge-1,▁▃▆█▇▇█
eval/rouge-2,▁▄▆█▇██
eval/rouge-l,▁▁▆█▇▇█
eval/runtime,▅▁▂█▁▁▆
eval/samples_per_second,▃█▇▁██▃
eval/steps_per_second,▃█▇▁██▃
train/epoch,▁▁▂▂▃▃▅▅▆▆▇▇███
train/global_step,▁▁▂▂▃▃▅▅▆▆▇▇███
+3,...


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]

./model_save/best_model 에 저장이 완료되었습니다.


## Inference Check

In [ ]:
# 1. 최고 성능 모델 & 토크나이저 불러오기
best_model_path = config['general']['output_dir'] + "Model_save/kobart_best_model"
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

print(f"{best_model_path} 에서 불러옵니다.")
infer_tokenizer = AutoTokenizer.from_pretrained(best_model_path)
infer_model = BartForConditionalGeneration.from_pretrained(best_model_path).to(device)

# 2. 검증 데이터 준비 (dev.csv)
data_path = config['general']['data_path']
val_df = pd.read_csv(os.path.join(data_path, 'dev_processed.csv'))

# 눈으로 확인 -> 앞의 3개 랑 뒤에 3개 데이터만 뽑아서 Test
sample_df = val_df.head(3)
sample_df = val_df.tail(3)
print("모델 추론 결과 확인해보기")
print("=" * 50)

# 3. 텍스트 생성
for idx, row in sample_df.iterrows():
    dialogue = row['dialogue']
    gold_summary = row['summary']

    # 규칙대로 프롬프트 결합 (대화문만)
    input_text = dialogue

    # 모델 입력 형태로 토크나이징
    inputs = infer_tokenizer(input_text, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)

    # Beam Search 등 모델 생성 옵션 적용
    summary_ids = infer_model.generate(
        inputs['input_ids'],
        num_beams=8,        # 수정사항 BeanSearch 값을 config[][].. 로 주는게 아닌 직접 숫자형을 매핑해서 깐깐하게
        max_length=50,
        early_stopping=True,
        no_repeat_ngram_size=2,
        # 수정사항 길이 패널티 추가 
        length_penalty=1.2,
        repetition_penalty=1.2
    )

    # 예측된 숫자를 다시 텍스트로 변환
    pred_summary = infer_tokenizer.decode(summary_ids[0], skip_special_tokens=False)

    # 클렌징
    remove_tokens = ['<usr>', '<s>', '</s>', '<pad>']
    for token in remove_tokens:
        pred_summary = pred_summary.replace(token, "").strip()

    print(f"[Sample {idx + 1}]")
    print(f"원본 대화 (앞부분 일부) : {dialogue[:150]}...")
    print(f"정답 요약 : {gold_summary}")
    print(f"예측 요약 : {pred_summary}")

You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.
You passed `num_labels=3` which is incompatible to the `id2label` map of length `2`.


./model_save/best_model_v9 에서 불러옵니다.


Loading weights: 100%|██████████| 260/260 [00:00<00:00, 7534.59it/s]


모델 추론 결과 확인해보기
[Sample 1]
원본 대화 (앞부분 일부) : [화자1]: 안녕하세요, 오늘 기분이 어떠세요?
[화자2]: 요즘 숨쉬기가 힘들어요.
[화자1]: 최근에 감기에 걸렸나요?
[화자2]: 아니요, 감기는 안 걸렸어요. 숨쉴 때 가슴이 답답해요.
[화자1]: 혹시 알고 있는 알레르기 있으세요?
[화자2]: 아니요, 특별히...
정답 요약 : [화자2]는 숨쉬기 어려워합니다. 의사는 [화자2]에게 증상을 확인하고, 천식 검사를 위해 폐 전문의에게 가볼 것을 권합니다.
예측 요약 : #Person2# 는 최근 숨쉬기 어려움을 겪고 있으며, #Person1# 은 천식 검사를 위해 폐 전문의에게 가볼 것을 권장합니다.
[Sample 2]
원본 대화 (앞부분 일부) : [화자1]: 야 Jimmy, 오늘 좀 이따 운동하러 가자.
[화자2]: 그래, 몇 시에 갈래?
[화자1]: 3시 30분 어때?
[화자2]: 좋아. 오늘은 다리랑 팔 운동하는 날이야.
[화자1]: 나 아까 농구해서 다리가 좀 아파. 오늘은 팔이랑 복근 운동하자.
[화자2...
정답 요약 : [화자1]는 Jimmy를 운동하러 초대하고 팔과 복근 운동을 하도록 설득합니다.
예측 요약 : #Person1# 과 Jimmy는 오늘 운동 계획을 세우고 #Person2# 에게 오후 3시 30분에 체육관에서 함께 운동하자고 제안합니다.
[Sample 3]
원본 대화 (앞부분 일부) : [화자1]: 나 건강에 안 좋은 음식 좀 그만 먹어야겠어. 
[화자2]: 맞아, 무슨 말인지 알아. 나도 요즘 건강하게 먹으려고 하거든. 
[화자1]: 요즘은 뭐 먹어? 
[화자2]: 주로 과일이랑 채소, 닭고기 먹지. 
[화자1]: 그게 다야? 
[화자2]: 거의 그...
정답 요약 : [화자1]은 건강에 안 좋은 음식을 그만 먹기로 결심하고, [화자2]는 자신의 건강한 식단을 [화자1]에게 공유합니다.
예측 요약 : #Person1# 과 #Person2# 는 건강한 식단에 대해 이야기하고 있습니다. 그들은 주로 

## Submission

In [ ]:
# 1. 테스트 데이터 불러오기 (대회 제공 test 파일)
test_df = pd.read_csv(os.path.join(config['general']['data_path'], 'test_processed.csv'))

# 2. 결과 저장을 위한 리스트
predicted_summaries = []

infer_model.eval()  # 평가모드 전환

# 3. 한 줄씩 추론 진행
with torch.no_grad():
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df)):
        # 훈련할 때 똑같은 하드 프롬프트 부착
        prompted_dialogue = row['dialogue']
        
        # 토크나이징 및 텐서 변환
        inputs = infer_tokenizer(prompted_dialogue, return_tensors="pt", padding=True, truncation=True, max_length=1024).to(device)

        # 찾아본 최적의 디코딩 파라미터 적용
        summary_ids = infer_model.generate(
            inputs['input_ids'],
            num_beams=8,
            length_penalty=1.2,
            repetition_penalty=1.2,
            max_length=50,
            early_stopping=True,
            no_repeat_ngram_size=2
        )

        # 디코딩 (skip_special_tokens=False 유지)
        pred_summary = infer_tokenizer.decode(summary_ids[0], skip_special_tokens=False)

        # 클렌징 (메모리 버그 방지로 인해 Config 에서 불러오는 것 보다 강제 하드코딩)
        remove_tokens_safe = ['<usr>', '<s>', '</s>', '<pad>']
        for token in remove_tokens_safe:
            pred_summary = pred_summary.replace(token, "").strip()

        predicted_summaries.append(pred_summary.strip())

# 4. 제출용 DataFrame 만들기 및 CSV Save
submission_df = pd.DataFrame({
    'fname': test_df['fname'],
    'summary': predicted_summaries
})

submission_path = '../Data/kobart_submission.csv'
submission_df.to_csv(submission_path, index=False, encoding='utf-8-sig')

print(f"제출 파일 생성 완료 : {submission_path}")

  0%|          | 0/499 [00:00<?, ?it/s]

100%|██████████| 499/499 [01:33<00:00,  5.33it/s]

제출 파일 생성 완료 : ../data/01_submission.csv
